# Wildfire Susceptibility Mapping — Selon la Methode Paper India
- Utilisation du dataSet reconstitué
- Spliting selon le ratio 1:1
- Entraînent des models
- Evaluation (Utilisation de Metric AUC et PR AUC pour comparaison future avec les autres methode implémenté)

1. Imports


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score,
    RocCurveDisplay, PrecisionRecallDisplay,
    precision_recall_curve, auc
)
from xgboost import XGBClassifier

2. Montage Google Drive

In [2]:
drive.mount('/content/gdrive')

Mounted at /content/gdrive


3. Chargement des données

In [3]:
csv_path  = "/content/gdrive/MyDrive/Projet big data dossier organisé /dataTraiter/datasetPaperIndian.csv"
dfDataSet = pd.read_csv(csv_path)

print("Dataset chargé :", dfDataSet.shape)
print(dfDataSet.columns.tolist())
dfDataSet.head()

Dataset chargé : (89198, 13)
['latitude', 'longitude', 'landcover', 'in_fire', 'dist_fire_km', 'mean_slope', 'mean_twi', 'mean_aspect', 'ndvi_final', 'mean_LST_celsius', 'dist_residentiel_m', 'dist_route_m', 'Elevation']


,latitude,longitude,landcover,in_fire,dist_fire_km,mean_slope,mean_twi,mean_aspect,ndvi_final,mean_LST_celsius,dist_residentiel_m,dist_route_m,Elevation
0,36.987823,-7.963676,210,0,17.197693,0.571321,6.447956,196.846283,0.686275,21.902494,1846.893734,471.899104,0.0
1,36.994129,-8.952213,110,0,23.004464,2.499209,3.489216,178.080078,0.866667,15.699432,1447.688717,170.582454,0.0
2,36.996850,-7.974890,210,0,16.054264,1.402498,7.057358,179.751862,0.889725,22.155281,457.222141,122.542919,0.0
3,37.003139,-8.952309,210,0,22.375234,2.263847,3.493453,290.926086,0.866667,16.453320,715.827701,364.845213,0.0
4,37.003216,-8.941076,210,0,21.596848,2.697602,4.360210,171.431244,0.918118,18.861568,293.616377,242.791353,0.0


4. Préparation & ratio 1:1

In [4]:
df_clean = dfDataSet.dropna(subset=['in_fire']).copy()

print("Distribution originale :")
print(df_clean['in_fire'].value_counts())
print(df_clean['in_fire'].value_counts(normalize=True))

# Équilibrage 1:1
fire            = df_clean[df_clean['in_fire'] == 1]
no_fire         = df_clean[df_clean['in_fire'] == 0]
no_fire_sampled = no_fire.sample(n=len(fire), random_state=42)
df_balanced     = pd.concat([fire, no_fire_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAprès équilibrage 1:1 :")
print(f"  Feux     : {len(fire)}")
print(f"  Non-feux : {len(no_fire_sampled)}")
print(f"  Total    : {len(df_balanced)}")

# Features et target
X = df_balanced.drop(columns=['in_fire', 'dist_fire_km', 'latitude', 'longitude'])
y = df_balanced['in_fire'].astype(int)
X = X.select_dtypes(include=[np.number])

print(f"\nFeatures utilisées ({len(X.columns)}) :")
print(list(X.columns))

Distribution originale :
in_fire
0    80336
1     8862
Name: count, dtype: int64
in_fire
0    0.900648
1    0.099352
Name: proportion, dtype: float64

Après équilibrage 1:1 :
  Feux     : 8862
  Non-feux : 8862
  Total    : 17724

Features utilisées (9) :
['landcover', 'mean_slope', 'mean_twi', 'mean_aspect', 'ndvi_final', 'mean_LST_celsius', 'dist_residentiel_m', 'dist_route_m', 'Elevation']


5. Split 70/30 + Prétraitement

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

y_train = np.array(y_train).astype(int)
y_test  = np.array(y_test).astype(int)

# Imputation médiane
imputer     = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),      columns=X.columns)

# Centrage + normalisation
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_test_sc  = scaler.transform(X_test_imp)

print(f"Train : {X_train_sc.shape[0]} échantillons ({X_train_sc.shape[0]/len(X)*100:.1f}%)")
print(f"Test  : {X_test_sc.shape[0]}  échantillons ({X_test_sc.shape[0]/len(X)*100:.1f}%)")

Train : 12406 échantillons (70.0%)
Test  : 5318  échantillons (30.0%)


6. Définition des modèles & configuration